In [1]:
import hashlib, json, math, statistics, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, memcap, sysinfo

B = repo / "build/adapters"
# Timing runs need the spec 7 conditions. caffeinate cannot stop clamshell sleep, so check the lid too.
assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
git = lambda *a: subprocess.run(["git", *a], cwd=repo, capture_output=True, text=True).stdout.strip()
run_id = "s4-cost-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
raw = repo / "results/raw" / run_id
raw.mkdir(parents=True)
snap = sysinfo.snapshot()
meta = {"run_id": run_id, "git_sha": git("rev-parse", "HEAD"), "dirty": bool(git("status", "--porcelain", "src", "Makefile")),
        "apir-adapter_sha256": hashlib.sha256((B / "apir-adapter").read_bytes()).hexdigest(), "sysinfo": snap}
(raw / "meta.json").write_text(json.dumps(meta, indent=1, default=str))
print({k: meta[k] for k in ("run_id", "git_sha", "dirty")})
print("power:", snap.get("power"), "| load:", snap.get("load"))

def run(name, args):
    """One S4 adapter process under caffeinate (spec 7) and the memory cap."""
    out = raw / f"{name}.jsonl"
    t = time.monotonic()
    with hygiene.Watch() as w:
        r = memcap.run(["caffeinate", "-dims", str(B / "apir-adapter"), *args, "--out", str(out)],
                       timeout_s=4 * 3600, cap_bytes=2 * memcap.GIB)
    (raw / f"{name}.stderr.log").write_text(r.stderr)
    with open(raw / "hygiene.jsonl", "a") as f:
        f.write(json.dumps({"run": name, **w.record()}) + "\n")
    assert not w.problems, f"{name}: not a valid timing run: {w.problems}"
    assert r.returncode == 0 and not r.over_cap, f"{name}: exit {r.returncode}, over_cap {r.over_cap}\n{r.stderr[-1500:]}"
    return [json.loads(l) for l in out.read_text().splitlines()], time.monotonic() - t

{'run_id': 's4-cost-20261110T164450Z', 'git_sha': '52afe4571c3d289fb952dafd0eb9f912c872c653', 'dirty': True}
power: {'source': 'AC Power', 'low_power_mode': False} | load: {'loadavg_1m': 4.30517578125, 'mem_available_bytes': 5693063168}


In [2]:
# S4 (DDH, P-256) at three sizes. Offline: one digest build (server_preproc_s). Online: 5 queries after 1 warm-up,
# each a fresh process that first rebuilds the digest untimed. Run with nothing else on the machine (spec 7).
sizes = {"64 KiB": 1 << 16, "256 KiB": 1 << 18, "1 MiB": 1 << 20}
common = ["--arm", "S4", "--record-bytes", "0", "--seed", "1234", "--fault", "none"]
rows = []
for label, db in sizes.items():
    off, off_wall = run(f"offline-{db}", [*common, "--db-bytes", str(db), "--phase", "offline", "--reps", "1", "--warmup", "0"])
    on, on_wall = run(f"online-{db}", [*common, "--db-bytes", str(db), "--phase", "online", "--reps", "5", "--warmup", "1"])
    timed = [r["metrics"] for r in on if not r["is_warmup"]]
    med = lambda k: statistics.median(m[k][0] if isinstance(m[k], list) else m[k] for m in timed)
    row = {"size": label, "db_bytes": db, "digest_s": off[0]["metrics"]["server_preproc_s"],
           "client_offline_s": off[0]["metrics"]["client_offline_s"],
           "query_gen_s": med("query_gen_s"), "server_answer_s": med("server_answer_s"),
           "client_recover_s": med("client_recover_s"), "online_queries": len(timed),
           "offline_process_wall_s": off_wall, "online_process_wall_s": on_wall}
    row["query_s"] = row["query_gen_s"] + row["server_answer_s"] + row["client_recover_s"]
    rows.append(row)
with open(raw / "s4_costs.jsonl", "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")
print(f"{'size':>8} {'digest s':>10} {'query s':>9} {'(gen':>8} {'answer':>8} {'recover)':>9} {'client offline s':>17}")
for r in rows:
    print(f"{r['size']:>8} {r['digest_s']:10.1f} {r['query_s']:9.2f} {r['query_gen_s']:8.3f} {r['server_answer_s']:8.2f} "
          f"{r['client_recover_s']:9.3f} {r['client_offline_s']:17.3f}")

    size   digest s   query s     (gen   answer  recover)  client offline s
  64 KiB       22.5      1.21    0.088     1.08     0.042             0.000
 256 KiB       85.8      4.56    0.176     4.30     0.083             0.000
   1 MiB      342.0     17.61    0.349    17.09     0.167             0.000


In [3]:
# Fit time = c * size^k on the three measured sizes (least squares on logs), then extrapolate to the spec's S4 sizes.
# A cell is one fresh process per phase (spec 9): offline = (5 reps + 1 warm-up) digests; online = one untimed
# digest for setup, then (30 reps + 1 warm-up) queries (spec 5.1 repetition rule). Cap: 30 min per cell (decision 10).
def fit(key):
    xs = [math.log(r["db_bytes"]) for r in rows]; ys = [math.log(r[key]) for r in rows]
    mx, my = statistics.mean(xs), statistics.mean(ys)
    k = sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / sum((x - mx) ** 2 for x in xs)
    return k, math.exp(my - k * mx)
fits = {key: fit(key) for key in ("digest_s", "query_s")}
for key, (k, c) in fits.items():
    print(f"{key}: grows as size^{k:.2f}")
OFF_DIGESTS, ON_QUERIES, CAP_MIN = 5 + 1, 30 + 1, 30
print(f"\n{'size':>8} {'source':>13} {'digest':>10} {'query':>9} {'offline cell':>13} {'online cell':>12}  within 30 min?")
out = []
for label, db in {"1 MiB": 1 << 20, "16 MiB": 1 << 24, "128 MiB": 1 << 27}.items():
    meas = next((r for r in rows if r["db_bytes"] == db), None)
    d, q = (meas["digest_s"], meas["query_s"]) if meas else tuple(c * db ** k for k, c in fits.values())
    off_min, on_min = OFF_DIGESTS * d / 60, (d + ON_QUERIES * q) / 60
    out.append({"size": label, "db_bytes": db, "source": "measured" if meas else "extrapolated", "digest_s": d,
                "query_s": q, "offline_cell_min": off_min, "online_cell_min": on_min})
    ok = lambda m: "yes" if m <= CAP_MIN else "no"
    print(f"{label:>8} {out[-1]['source']:>13} {d / 60:8.1f} m {q:7.1f} s {off_min:10.0f} min {on_min:9.0f} min  "
          f"offline {ok(off_min)}, online {ok(on_min)}")
with open(raw / "s4_cells.jsonl", "w") as f:
    for row in out:
        f.write(json.dumps(row) + "\n")

digest_s: grows as size^0.98
query_s: grows as size^0.97

    size        source     digest     query  offline cell  online cell  within 30 min?
   1 MiB      measured      5.7 m    17.6 s         34 min        15 min  offline no, online yes
  16 MiB  extrapolated     86.2 m   255.3 s        517 min       218 min  offline no, online no
 128 MiB  extrapolated    664.2 m  1902.3 s       3985 min      1647 min  offline no, online no
